# VeriGap — a Kaggle Benchmark (`kbench`)

**The verification gap:** can a model, given only a program *specification*, write a test suite strong enough to tell a correct implementation from a subtly broken (mutant) one?

- The task prompts the model to generate a test suite from the spec, then scores it **without Docker**.
- **Metric (leaderboard):** `mutation_score` = fraction of mutants killed (a mutant is killed if any reference-passing test fails on it).
- **Test all models:** written against `kbench.llm`, so use the **Add Models** button on the task page — or pass a list to `.evaluate()` below.
- Data (reference binary + mutants) is pulled from the public `eth-sri/tsb-*` HF datasets at runtime; binaries run natively on Kaggle's Linux.

In [ ]:
import os, sys, shutil, subprocess, tempfile, gzip, hashlib, json, stat, base64
from pathlib import Path
from dataclasses import dataclass
import xml.etree.ElementTree as ET
try:
    import kaggle_benchmarks as kbench
except ImportError:
    # pre-installed in Kaggle's Benchmarks runtime (kaggle.com/benchmarks/tasks/new); fall back otherwise
    subprocess.run([sys.executable,'-m','pip','install','-q','kaggle-benchmarks'])
    import kaggle_benchmarks as kbench
subprocess.run(['pip','install','-q','huggingface_hub','pytest'], capture_output=True)
from huggingface_hub import snapshot_download
WS = Path('/workspace')
if not shutil.which('python'):
    os.symlink(shutil.which('python3'), '/usr/local/bin/python')

In [ ]:
# ---- data + container-free scoring helpers (reused from the validated pipeline) ----
def fetch_task(task):
    impl = Path(snapshot_download('eth-sri/tsb-reference_impl', repo_type='dataset', allow_patterns=f'executable/{task}/**'))
    ref = impl/'executable'/task/'executable'; ref.chmod(0o755)
    mut_root = Path(snapshot_download('eth-sri/tsb-mutants', repo_type='dataset', allow_patterns=f'{task}/**'))/task
    manifest = json.loads((mut_root/'mutants.json').read_text())
    mutants = manifest['mutants'] if isinstance(manifest, dict) else manifest
    return ref, mut_root, mutants

def materialize_mutant(mut_root, m):
    payload = mut_root/m['payload']
    if payload.is_file():
        return payload, False
    art = m.get('artifact');
    if not art: raise FileNotFoundError(payload)
    fd, name = tempfile.mkstemp(prefix='mut-'); os.close(fd); tmp = Path(name)
    with gzip.open(mut_root/art['path'], 'rb') as c, open(tmp, 'wb') as o:
        shutil.copyfileobj(c, o, length=1<<20)
    assert hashlib.sha256(tmp.read_bytes()).hexdigest() == art['sha256'], 'hash mismatch'
    tmp.chmod(tmp.stat().st_mode | stat.S_IXUSR)
    return tmp, True

def run_against(exe, suite_root):
    if WS.exists(): shutil.rmtree(WS)
    WS.mkdir(parents=True)
    shutil.copy2(exe, WS/'executable'); (WS/'executable').chmod(0o755)
    for item in suite_root.iterdir():
        dst = WS/item.name
        shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)
    try:
        subprocess.run(['bash','eval/run.sh'], cwd=WS, capture_output=True, text=True, timeout=1800)
    except subprocess.TimeoutExpired:
        pass
    res = WS/'eval'/'results.xml'; out = {}
    if res.is_file():
        for tc in ET.parse(res).getroot().iter('testcase'):
            name = (tc.get('classname','')+'::'+tc.get('name','')).strip(':')
            tags = {c.tag for c in tc}
            out[name] = 'skipped' if 'skipped' in tags else ('failed' if ({'failure','error'} & tags) else 'passed')
    return out

In [ ]:
@dataclass
class TestFile:
    path: str
    content: str

@dataclass
class GeneratedSuite:
    files: list

INSTRUCTIONS = 'You are creating a test suite.\n\n## Task\n\nCreate a comprehensive test suite under `eval/`.\n\n## Evaluation Design\n\n- The suite goes through mutation testing against fresh workspaces containing the program under test at `/workspace/executable`.\n- The specification is available only now, while authoring the suite.\n- Tests must invoke `/workspace/executable` (absolute path).\n\n## Rules\n\n- Cover documented functional behavior thoroughly. The suite must kill mutants that violate the spec, while passing any implementation that follows the spec.\n- Cover each explicitly numbered/sectioned requirement with at least one focused test. Prefer many small tests over one huge test.\n\n## Output\n\nReturn a list of files. Exactly one must be `eval/run.sh` (executable), plus any test files it needs.\n- `eval/run.sh` must always write valid JUnit XML to `/workspace/eval/results.xml`.\n- It must run the whole suite even after individual failures.\n- The report root is `<testsuites>` (or a single `<testsuite>`), one named `<testcase>` per test; non-pass outcomes use nested `<failure>`/`<error>`/`<skipped>`.\n- For pytest: `cd /workspace/eval && pytest --junitxml=/workspace/eval/results.xml` achieves this.\n'

In [ ]:
@kbench.task(name='verigap')
def verigap(llm, task: str, spec: str) -> float:
    """Generate a test suite from the spec, score mutation kill rate (container-free)."""
    suite = llm.prompt(INSTRUCTIONS + '\n\n## Specification\n\n' + spec, schema=GeneratedSuite)
    root = Path(tempfile.mkdtemp())/'suite'; root.mkdir(parents=True)
    for f in suite.files:
        p = root/f.path; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(f.content)
        if p.name == 'run.sh': p.chmod(0o755)
    ref, mut_root, mutants = fetch_task(task)
    ref_out = run_against(ref, root)
    gt_pass = {n for n, s in ref_out.items() if s == 'passed'}
    kbench.assertions.assert_true(bool(gt_pass),
        expectation='suite must produce at least one test that passes on the reference implementation')
    if not gt_pass:
        return 0.0
    killed = 0
    for m in mutants:
        try:
            mexe, temp = materialize_mutant(mut_root, m)
        except Exception:
            continue
        try:
            out = run_against(mexe, root)
            if any(out.get(n) != 'passed' for n in gt_pass):
                killed += 1
        finally:
            if temp: Path(mexe).unlink(missing_ok=True)
    return killed/len(mutants) if mutants else 0.0

## Run

Below runs one task against the default model. To **test all models**, use the **Add Models** button on the task detail page (no code change), or uncomment the multi-model list. To add tasks, append rows to `df` (each needs `task` + its `spec` text).

In [ ]:
import pandas as pd
SPEC = base64.b64decode('IyBDTWF0cml4IGV4ZWN1dGFibGUgc3BlY2lmaWNhdGlvbgoKVGhlIHJlcXVpcmVkIGFydGlmYWN0IGlzIGEgZGlyZWN0bHkgcnVubmFibGUgZmlsZSBuYW1lZCBleGFjdGx5IGBleGVjdXRhYmxlYC4gSXQgaXMgYSB0ZXJtaW5hbCBwcm9ncmFtIHdoaWNoIHByZXNlbnRzIGEgTWF0cml4LXN0eWxlIGFuaW1hdGVkIGNoYXJhY3RlciByYWluLiBUaGUgcmVxdWlyZW1lbnRzIGJlbG93IGRlc2NyaWJlIGl0cyBjb21wbGV0ZSBleHRlcm5hbGx5IG9ic2VydmFibGUgY29udHJhY3QuCgojIyBTcGVjIDEKCmBleGVjdXRhYmxlYCBpbXBsZW1lbnRzIGEgdHJhZGl0aW9uYWwsIHRvcC1sZXZlbCBVbml4IHNob3J0LW9wdGlvbiBpbnRlcmZhY2U7IGl0IGhhcyBubyBzdWJjb21tYW5kcy4gSW52b2NhdGlvbnMgc3VjaCBhcyBgYWRkYCwgYHJlbW92ZWAsIGBsaXN0YCwgYHNob3dgLCBgaW5pdGAsIGBoZWxwYCwgYHZlcnNpb25gLCBvciBhbnkgb3RoZXIgbm9uLW9wdGlvbiB3b3JkIGFyZSBub3QgY29tbWFuZHMuIFBvc2l0aW9uYWwgb3BlcmFuZHMgYXJlIG90aGVyd2lzZSB0b2xlcmF0ZWQgYW5kIGlnbm9yZWQsIHNvIGlmIHRlcm1pbmFsIGluaXRpYWxpemF0aW9uIGlzIHBvc3NpYmxlIHRoZSBwcm9ncmFtIHByb2NlZWRzIHRvIGl0cyBhbmltYXRpb24uIEEgc3RhbmRhbG9uZSBgLS1gIGVuZHMgb3B0aW9uIHByb2Nlc3NpbmcsIGFuZCBsYXRlciB3b3JkcyBhcmUgcG9zaXRpb25hbCBvcGVyYW5kcy4KClNob3J0IG9wdGlvbnMgbWF5IGJlIHNlcGFyYXRlIG9yIGdyb3VwZWQsIHNvIGZvcm1zIHN1Y2ggYXMgYC1hIC1iYCwgYC1hYmAsIGAtYWJjYCwgYC1sYmFgLCBhbmQgYC1hYm5WYCBhcmUgdmFsaWQuIE9wdGlvbnMgd2hpY2ggdGFrZSBhIHZhbHVlIGFjY2VwdCBlaXRoZXIgYSBmb2xsb3dpbmcgYXJndW1lbnQgb3IgYW4gYXR0YWNoZWQgdmFsdWUgd2hlcmUgY29udmVudGlvbmFsIHNob3J0LW9wdGlvbiBwYXJzaW5nIHBlcm1pdHMgaXQ7IGZvciBleGFtcGxlLCBib3RoIGAtdSA0YCBhbmQgYC11NGAgd29yay4gUmVwZWF0aW5nIGFuIG9wdGlvbiBpcyBzYWZlLiBFeGNlcHQgZm9yIHRoZSB0ZXJtaW5hdGluZyBpbmZvcm1hdGlvbmFsL2Vycm9yIGNhc2VzIGRlc2NyaWJlZCBiZWxvdywgb3B0aW9ucyBjYW4gYmUgZnJlZWx5IGludGVybGVhdmVkLgoKIyMgU3BlYyAyCgpUaGUgZm9sbG93aW5nIGludm9jYXRpb25zIHByaW50IHRoZSBjYW5vbmljYWwgaGVscCB0ZXh0IGZyb20gU3BlYyAzIHRvIHN0YW5kYXJkIG91dHB1dCBhbmQgdGVybWluYXRlIHN1Y2Nlc3NmdWxseSB3aXRoIHN0YXR1cyAwOgoKLSBgLWhgLCBgLT9gLCBhbmQgYC0taGVscGA7Ci0gYW55IHVua25vd24gb3B0aW9uLCBpbmNsdWRpbmcgdW5rbm93biBzaG9ydCBvcHRpb25zIHN1Y2ggYXMgYC16YCBvciBgLVpgIGFuZCB1bmtub3duIGxvbmcgb3B0aW9ucyBzdWNoIGFzIGAtLXZlcnNpb25gIG9yIGAtLW5vbmV4aXN0ZW50YDsKLSBhIHZhbHVlLXRha2luZyBvcHRpb24gd2hpY2ggaGFzIG5vIHZhbHVlOiBgLUNgLCBgLXVgLCBgLU1gLCBvciBgLXRgIGF0IHRoZSBlbmQgb2YgdGhlIG9wdGlvbiBsaXN0LgoKVGhlc2UgY2FzZXMgcHJvZHVjZSBubyBkaWFnbm9zdGljIG9uIHN0YW5kYXJkIGVycm9yLiBIZWxwIGFuZCB2ZXJzaW9uIG9wdGlvbnMgYXJlIHByb2Nlc3NlZCBmcm9tIGxlZnQgdG8gcmlnaHQgYW5kIHRlcm1pbmF0ZSBwcm9jZXNzaW5nIGltbWVkaWF0ZWx5OiBmb3IgZXhhbXBsZSwgYC1WIC1oYCBwcmludHMgdmVyc2lvbiBpbmZvcm1hdGlvbiwgd2hpbGUgYC1oIC1WYCBwcmludHMgaGVscC4gUmVjb2duaXplZCBub24tdGVybWluYXRpbmcgZmxhZ3MgbWF5IHByZWNlZGUgZWl0aGVyIG9uZS4gSGVscC92ZXJzaW9uIG91dHB1dCBtdXN0IG5vdCBpbml0aWFsaXplIHRoZSB0ZXJtaW5hbCBVSS4KCiMjIFNwZWMgMwoKVGhlIGNhbm9uaWNhbCBoZWxwIG91dHB1dCBpcyBleGFjdGx5IHRoZSBmb2xsb3dpbmcgMjEgbGluZXMsIGluY2x1ZGluZyB0aGUgc2luZ2xlIGxlYWRpbmcgc3BhY2Ugb24gZXZlcnkgbGluZSwgY2FwaXRhbGl6YXRpb24sIHB1bmN0dWF0aW9uLCBsaW5lIGJyZWFrcywgYW5kIG9uZSBmaW5hbCBuZXdsaW5lLiBJdCBjb250YWlucyBubyB0ZXJtaW5hbC1jb250cm9sIG9yIEFOU0kgZXNjYXBlIHNlcXVlbmNlcy4KCmBgYHRleHQKIFVzYWdlOiBjbWF0cml4IC1bYWJCY2ZobHNtVnhrXSBbLXUgZGVsYXldIFstQyBjb2xvcl0gWy10IHR0eV0gWy1NIG1lc3NhZ2VdCiAtYTogQXN5bmNocm9ub3VzIHNjcm9sbAogLWI6IEJvbGQgY2hhcmFjdGVycyBvbgogLUI6IEFsbCBib2xkIGNoYXJhY3RlcnMgKG92ZXJyaWRlcyAtYikKIC1jOiBVc2UgSmFwYW5lc2UgY2hhcmFjdGVycyBhcyBzZWVuIGluIHRoZSBvcmlnaW5hbCBtYXRyaXguIFJlcXVpcmVzIGFwcHJvcHJpYXRlIGZvbnRzCiAtZjogRm9yY2UgdGhlIGxpbnV4ICRURVJNIHR5cGUgdG8gYmUgb24KIC1sOiBMaW51eCBtb2RlICh1c2VzIG1hdHJpeCBjb25zb2xlIGZvbnQpCiAtTDogTG9jayBtb2RlIChjYW4gYmUgY2xvc2VkIGZyb20gYW5vdGhlciB0ZXJtaW5hbCkKIC1vOiBVc2Ugb2xkLXN0eWxlIHNjcm9sbGluZwogLWg6IFByaW50IHVzYWdlIGFuZCBleGl0CiAtbjogTm8gYm9sZCBjaGFyYWN0ZXJzIChvdmVycmlkZXMgLWIgYW5kIC1CLCBkZWZhdWx0KQogLXM6ICJTY3JlZW5zYXZlciIgbW9kZSwgZXhpdHMgb24gZmlyc3Qga2V5c3Ryb2tlCiAteDogWCB3aW5kb3cgbW9kZSwgdXNlIGlmIHlvdXIgeHRlcm0gaXMgdXNpbmcgbXR4LnBjZgogLVY6IFByaW50IHZlcnNpb24gaW5mb3JtYXRpb24gYW5kIGV4aXQKIC1NIFttZXNzYWdlXTogUHJpbnRzIHlvdXIgbWVzc2FnZSBpbiB0aGUgY2VudGVyIG9mIHRoZSBzY3JlZW4uIE92ZXJyaWRlcyAtTCdzIGRlZmF1bHQgbWVzc2FnZS4KIC11IGRlbGF5ICgwIC0gMTAsIGRlZmF1bHQgNCk6IFNjcmVlbiB1cGRhdGUgZGVsYXkKIC1DIFtjb2xvcl06IFVzZSB0aGlzIGNvbG9yIGZvciBtYXRyaXggKGRlZmF1bHQgZ3JlZW4pCiAtcjogcmFpbmJvdyBtb2RlCiAtbTogbGFtYmRhIG1vZGUKIC1rOiBDaGFyYWN0ZXJzIGNoYW5nZSB3aGlsZSBzY3JvbGxpbmcuIChXb3JrcyB3aXRob3V0IC1vIG9wdC4pCiAtdCBbdHR5XTogU2V0IHR0eSB0byB1c2UKYGBgCgojIyBTcGVjIDQKCmAtVmAgcHJpbnRzIHZlcnNpb24gaW5mb3JtYXRpb24gdG8gc3RhbmRhcmQgb3V0cHV0LCBwcm9kdWNlcyBub3RoaW5nIG9uIHN0YW5kYXJkIGVycm9yLCBleGl0cyB3aXRoIHN0YXR1cyAwLCBhbmQgZG9lcyBub3QgaW5pdGlhbGl6ZSB0aGUgVUkuIEl0IHN0YXJ0cyB3aXRoIGEgc2luZ2xlIGxlYWRpbmcgc3BhY2UgYW5kIGhhcyB0aGlzIHRocmVlLWxpbmUgc2hhcGUsIHdpdGggYSBmaW5hbCBuZXdsaW5lOgoKYGBgdGV4dAogQ01hdHJpeCB2ZXJzaW9uIDIuMCAoY29tcGlsZWQgSEg6TU06U1MsIE1vbiBERCBZWVlZKQpFbWFpbDogYWJpc2hla3Zhc2hva0BnbWFpbC5jb20KV2ViOiBodHRwczovL2dpdGh1Yi5jb20vYWJpc2hla3Zhc2hvay9jbWF0cml4CmBgYAoKVGhlIHRleHQgYEhIOk1NOlNTLCBNb24gREQgWVlZWWAgZGVub3RlcyBhIHJlYWwgY29tcGlsZS9idWlsZCB0aW1lc3RhbXAgcmVuZGVyZWQgaW4gdGhlIGNvbnZlbnRpb25hbCBDIGRhdGUvdGltZSBzdHlsZSAoZm9yIGV4YW1wbGUsIGAyMDoxNToyNiwgTWFyICA1IDIwMjZgOyBhIG9uZS1kaWdpdCBkYXkgaXMgc3BhY2UtcGFkZGVkKS4gVGhlIGludmFyaWFudCBmaWVsZHMgYXJlIHRoZSBwcm9kdWN0IG5hbWUgYENNYXRyaXhgLCB2ZXJzaW9uIGAyLjBgLCB0aGUgZXhhY3QgZW1haWwgYWRkcmVzcywgYW5kIHRoZSBleGFjdCBIVFRQUyBwcm9qZWN0IFVSTC4gVGhlIHZlcnNpb24gb3V0cHV0IG11c3QgY29udGFpbiB0aGUgd29yZCBgY29tcGlsZWRgIGFuZCBtdXN0IGRpZmZlciBmcm9tIHRoZSBoZWxwIHRleHQuCgojIyBTcGVjIDUKCkFmdGVyIHBhcnNpbmcgYSBub3JtYWwgaW52b2NhdGlvbiB3aGljaCBkb2VzIG5vdCBjb250YWluIGAtbGAsIHRoZSBwcm9ncmFtIGluaXRpYWxpemVzIGEgdGVybWluYWwtY2FwYWJpbGl0eS1iYXNlZCBmdWxsLXNjcmVlbiBpbnRlcmZhY2UuIFRoZSBgLWxgIExpbnV4LWZvbnQgcGF0aCBpcyB0aGUgdGVybWluYXRpbmcgc3BlY2lhbCBjYXNlIHNwZWNpZmllZCBpbiBTcGVjIDguIENvbW1vbiB0ZXJtaW5hbCB0eXBlcyBpbmNsdWRpbmcgYHh0ZXJtYCwgYHh0ZXJtLTI1NmNvbG9yYCwgYGxpbnV4YCwgYHZ0MTAwYCwgYGR1bWJgLCBhbmQgYHNjcmVlbmAgbXVzdCBub3QgYWZmZWN0IGhlbHAgb3IgdmVyc2lvbiBvdXRwdXQuIGAtZmAgc2VsZWN0cyBMaW51eCB0ZXJtaW5hbCBiZWhhdmlvciBpbnRlcm5hbGx5IGFuZCBpcyBhY2NlcHRlZCByZWdhcmRsZXNzIG9mIHRoZSBpbmhlcml0ZWQgYFRFUk1gIHZhbHVlLgoKSWYgYSBub3JtYWwgaW52b2NhdGlvbiBjYW5ub3QgaW5pdGlhbGl6ZSB0aGUgdGVybWluYWwgYmVjYXVzZSB0aGUgdGVybWluYWwgdHlwZSBpcyBtaXNzaW5nIG9yIHVudXNhYmxlLCBpdCBtdXN0IGZhaWwgcHJvbXB0bHkgd2l0aCBub256ZXJvIHN0YXR1cyBhbmQgYSBkaWFnbm9zdGljIG9mIHRoZSBmb3JtIGBFcnJvciBvcGVuaW5nIHRlcm1pbmFsOiBUWVBFLmAgZm9sbG93ZWQgYnkgYSBuZXdsaW5lLiBXaGVuIG5vIHVzYWJsZSB0eXBlIGlzIGF2YWlsYWJsZSwgdGhlIG9ic2VydmFibGUgZGlhZ25vc3RpYyBpcyBleGFjdGx5IGBFcnJvciBvcGVuaW5nIHRlcm1pbmFsOiB1bmtub3duLmAuIFRoZSBkaWFnbm9zdGljIG1heSBiZSB3cml0dGVuIHRvIHN0YW5kYXJkIGVycm9yLiBEbyBub3QgcmVqZWN0IGFuIGludm9jYXRpb24gbWVyZWx5IGJlY2F1c2UgaXRzIG91dHB1dCBpcyBjYXB0dXJlZCByYXRoZXIgdGhhbiBhdHRhY2hlZCB0byBhIHBoeXNpY2FsIHRlcm1pbmFsOiB3aXRoIGEgc3VwcG9ydGVkIGBURVJNYCBzdWNoIGFzIGB4dGVybWAsIGl0IG11c3QgZW50ZXIgdGhlIHJlbmRlciBsb29wIGFuZCBjb250aW51ZSB1bnRpbCBpbnRlcnJ1cHRlZC4KCiMjIFNwZWMgNgoKT24gYSBjb21wYXRpYmxlIHRlcm1pbmFsLCB0aGUgVUkgc3dpdGNoZXMgdG8gdGhlIGFsdGVybmF0ZSBzY3JlZW4sIGhpZGVzIHRoZSBjdXJzb3IsIGFuZCBiZWdpbnMgZHJhd2luZyB3aXRob3V0IHJlcXVpcmluZyBpbnB1dC4gRm9yIGFuIHh0ZXJtLWxpa2UgdGVybWluYWwgdGhlIGVtaXR0ZWQgc2V0dXAgc3RyZWFtIGluY2x1ZGVzIGBFU0MgWz8xMDQ5aGAgYW5kIGBFU0MgWz8yNWxgIHdpdGhvdXQgdGhlIHNwYWNlcyBzaG93biBoZXJlICh0aGF0IGlzLCBieXRlIHN0cmluZ3MgYFx4MWJbPzEwNDloYCBhbmQgYFx4MWJbPzI1bGApLiBPbiBub3JtYWwgb3IgaW50ZXJydXB0ZWQgc2h1dGRvd24gaXQgcmVzdG9yZXMgdGVybWluYWwgc3RhdGUsIG1ha2VzIHRoZSBjdXJzb3IgdXNhYmxlIGFnYWluLCBhbmQgbGVhdmVzIHRoZSBhbHRlcm5hdGUgc2NyZWVuOyB0aGUgeHRlcm0tY29tcGF0aWJsZSBvdXRwdXQgaW5jbHVkZXMgYFx4MWJbPzEwNDlsYC4KCkV4Y2VwdCBmb3IgdGhlIHRlcm1pbmF0aW5nIGAtbGAgYmVoYXZpb3IgaW4gU3BlYyA4LCB0aGUgcHJvY2VzcyByZW1haW5zIHJ1bm5pbmcgaW4gaXRzIHJlbmRlciBsb29wIHJhdGhlciB0aGFuIGV4aXRpbmcgYnkgaXRzZWxmLiBJbiBvcmRpbmFyeSwgbm9uLWxvY2tlZCBtb2RlcywgcmVjZWlwdCBvZiBhbiBpbnRlcnJ1cHQgc3VjaCBhcyBDdHJsLUMvU0lHSU5UIHRlcm1pbmF0ZXMgaXQgcHJvbXB0bHkgYWZ0ZXIgcmVzdG9yaW5nIHRoZSB0ZXJtaW5hbC4gTG9ja2VkIG1vZGUgKGAtTGApIGRlbGliZXJhdGVseSBpZ25vcmVzIFNJR0lOVDogYWZ0ZXIgcmVjZWl2aW5nIGl0LCB0aGUgcHJvY2VzcyBjb250aW51ZXMgcmVuZGVyaW5nIGFuZCByZW1haW5zIGFsaXZlIGZvciBsb25nZXIgdGhhbiB0d28gc2Vjb25kcy4gVGhpcyBhcHBsaWVzIGJvdGggdG8gYC1MYCBhbG9uZSBhbmQgdG8gbG9ja2VkIG1vZGUgd2l0aCBlaXRoZXIgdGhlIGRlZmF1bHQgb3IgYSBjdXN0b20gYC1NYCBtZXNzYWdlLgoKIyMgU3BlYyA3CgpUaGUgZnVsbC1zY3JlZW4gcHJlc2VudGF0aW9uIGlzIGEgY29udGludW91c2x5IGNoYW5naW5nIE1hdHJpeC1yYWluIGFuaW1hdGlvbiwgbm90IGEgc3RhdGljIGJhbm5lci4gSXQgZmlsbHMgdGhlIHNjcmVlbiB3aXRoIG1hbnkgbm9uLXNwYWNlIGdseXBocyBhcnJhbmdlZCBpbiB2ZXJ0aWNhbGx5IG1vdmluZyBzdHJlYW1zLCB3aXRoIGJsYW5rIGdhcHMgYmV0d2VlbiBzdHJlYW1zLiBBIHR5cGljYWwgODAtYnktMjQgb3IgMTIwLWJ5LTQwIHRlcm1pbmFsIG11c3Qgc2hvdyBzdWJzdGFudGlhbGx5IG1vcmUgdGhhbiBhIHRva2VuIGFtb3VudCBvZiBvdXRwdXQgd2l0aGluIGFib3V0IDMwMCBtcywgYW5kIGNhcHR1cmVkIHBhbmUgY29udGVudHMgbXVzdCBjaGFuZ2UgYWdhaW4gd2l0aGluIGEgZnJhY3Rpb24gb2YgYSBzZWNvbmQuIEFuaW1hdGlvbiBjb250aW51ZXMgaW5kZWZpbml0ZWx5IHVudGlsIGFuIGFwcGxpY2FibGUgZXhpdCBldmVudC4KCkdseXBocyBtYXkgYmUgcmFuZG9taXplZCBvbiBlYWNoIHJ1bi4gT3JkaW5hcnkgbW9kZSB1c2VzIE1hdHJpeC1saWtlIGxldHRlcnMsIGRpZ2l0cywgc3ltYm9scywgYW5kL29yIEphcGFuZXNlLXN0eWxlIGdseXBocywgd2l0aCBicmlnaHRlciBzdHJlYW0gaGVhZHMgYW5kIGRpbW1lciB0YWlscy4gRXZlcnkgbm9ybWFsIG9wdGlvbiBjb21iaW5hdGlvbiB3aGljaCBkb2VzIG5vdCBjb250YWluIGAtbGAgbXVzdCBjb250aW51ZSB0byBwcm9kdWNlIGEgbm9uZW1wdHkgY2hhbmdpbmcgc2NyZWVuOyBjb21iaW5hdGlvbnMgY29udGFpbmluZyBgLWxgIGluc3RlYWQgZm9sbG93IHRoZSBkZXRlcm1pbmlzdGljIHRlcm1pbmF0aW5nIHBhdGggaW4gU3BlYyA4LgoKIyMgU3BlYyA4CgpUaGUgYm9vbGVhbiBkaXNwbGF5IG9wdGlvbnMgYXJlIGFjY2VwdGVkIGJvdGggc2VwYXJhdGVseSBhbmQgaW4gZ3JvdXBlZCBzaG9ydC1vcHRpb24gZm9ybToKCi0gYC1hYCBzZWxlY3RzIGFzeW5jaHJvbm91cyBzdHJlYW1zLCB3aG9zZSBjb2x1bW5zIGFkdmFuY2UgaW5kZXBlbmRlbnRseTsKLSBgLW9gIHNlbGVjdHMgb2xkLXN0eWxlIHNjcm9sbGluZzsKLSBgLWNgIHNlbGVjdHMgdGhlIG9yaWdpbmFsL0phcGFuZXNlIGNoYXJhY3RlciByZXBlcnRvaXJlOwotIGAtcmAgc2VsZWN0cyByYWluYm93IGNvbG9yaW5nOwotIGAtbWAgc2VsZWN0cyBsYW1iZGEgbW9kZSwgdXNpbmcgbGFtYmRhIGdseXBocyBmb3IgdGhlIHJhaW47Ci0gYC1rYCBtYWtlcyBhbHJlYWR5LXZpc2libGUgY2hhcmFjdGVycyBjaGFuZ2Ugd2hpbGUgdGhleSBzY3JvbGw7Ci0gYC1sYCByZXF1ZXN0cyBMaW51eC1jb25zb2xlL2ZvbnQgbW9kZTsKLSBgLXhgIHNlbGVjdHMgWC13aW5kb3cgZm9udC9sYXlvdXQgbW9kZTsKLSBgLWZgIGZvcmNlcyBMaW51eCB0ZXJtaW5hbCBoYW5kbGluZzsKLSBgLXNgIHNlbGVjdHMgc2NyZWVuc2F2ZXIgaW5wdXQgYmVoYXZpb3I7Ci0gYC1MYCBzZWxlY3RzIHRoZSBsb2NrZWQgcHJlc2VudGF0aW9uLgoKVGhlIG9wdGlvbnMgYXJlIHN5bnRhY3RpY2FsbHkgY29tcG9zYWJsZS4gQ29tYmluYXRpb25zIHdoaWNoIGRvIG5vdCBjb250YWluIGAtbGAsIHN1Y2ggYXMgYXN5bmNocm9ub3VzIHBsdXMgYm9sZCBwbHVzIHJhaW5ib3cgb3IgY29sb3IgcGx1cyB1cGRhdGUgZGVsYXksIGVudGVyIHRoZSBsaXZlIHJlbmRlciBsb29wIHdpdGhvdXQgYW4gb3B0aW9uIGVycm9yLgoKSW4gdGhpcyBleGVjdXRhYmxlLCBldmVyeSBub3JtYWwgaW52b2NhdGlvbiBjb250YWluaW5nIGAtbGAgdGFrZXMgYSBkZXRlcm1pbmlzdGljIGZvbnQtdG9vbCBjb25mbGljdCBwYXRoIHJhdGhlciB0aGFuIHJlbmRlcmluZyByYWluLiBJdCBlbnRlcnMgYW5kIHRoZW4gY2xlYW5zIHVwIHRoZSB0ZXJtaW5hbCBVSSwgd3JpdGVzIGV4YWN0bHkgdGhlIGZvbGxvd2luZyBvbmUtbGluZSBkaWFnbm9zdGljIHRvIHN0YW5kYXJkIGVycm9yIChpbmNsdWRpbmcgaXRzIGxlYWRpbmcgc3BhY2UgYW5kIGZpbmFsIG5ld2xpbmUpLCBwcm9kdWNlcyB0ZXJtaW5hbCBzZXR1cC9jbGVhbnVwIGNvbnRyb2wgYnl0ZXMgb24gc3RhbmRhcmQgb3V0cHV0LCBhbmQgZXhpdHMgd2l0aCBzdGF0dXMgMDoKCmBgYHRleHQKIFVuYWJsZSB0byB1c2UgYm90aCAic2V0Zm9udCIgYW5kICJjb25zb2xlY2hhcnMiLgpgYGAKClRoaXMgYmVoYXZpb3IgYXBwbGllcyB0byBgLWxgIGJ5IGl0c2VsZiBhbmQgd2hlbiBpdCBpcyBncm91cGVkIHdpdGggb3RoZXIgZGlzcGxheSBmbGFncy4gSW4gcGFydGljdWxhciwgYC1sYmFgLCBgLW9sYCwgYW5kIGAtYW9iQm5rY3JtbHhgIGFsbCBmb2xsb3cgdGhpcyBzYW1lIGVhcmx5LWV4aXQgcGF0aCBhbmQgbmV2ZXIgYmVnaW4gdGhlIGFuaW1hdGlvbi4KCiMjIFNwZWMgOQoKQ2hhcmFjdGVyIGludGVuc2l0eSBoYXMgdGhyZWUgbW9kZXMuIFRoZSBkZWZhdWx0IGFuZCBgLW5gIHVzZSBubyByYW5kb20gYm9sZCBoaWdobGlnaHRpbmc7IGAtYmAgZW5hYmxlcyBvY2Nhc2lvbmFsIGJvbGQgY2hhcmFjdGVyczsgYC1CYCBtYWtlcyBhbGwgcmFpbiBjaGFyYWN0ZXJzIGJvbGQuIGAtQmAgc3VwZXJzZWRlcyBgLWJgLCBhbmQgYC1uYCBzdXBlcnNlZGVzIGJvdGggYC1iYCBhbmQgYC1CYC4gQ29tYmluaW5nIG9yIHJlcGVhdGluZyB0aGVzZSBvcHRpb25zIG11c3QgcmVtYWluIHZhbGlkLiBXaGVuIGZvbGxvd2VkIGJ5IGAtVmAgb3IgYC1oYCwgZXZlcnkgc3VjaCBjb21iaW5hdGlvbiB0ZXJtaW5hdGVzIHdpdGggdGhlIHJlcXVlc3RlZCBpbmZvcm1hdGlvbmFsIG91dHB1dCByYXRoZXIgdGhhbiBzdGFydGluZyB0aGUgVUkuCgojIyBTcGVjIDEwCgpgLUMgQ09MT1JgIGNob29zZXMgdGhlIGJhc2UgcmFpbiBjb2xvci4gVGhlIGFjY2VwdGVkIG5hbWVzIGFyZSBgZ3JlZW5gLCBgcmVkYCwgYGJsdWVgLCBgd2hpdGVgLCBgeWVsbG93YCwgYGN5YW5gLCBgbWFnZW50YWAsIGFuZCBgYmxhY2tgOyBtYXRjaGluZyBpcyBjYXNlLWluc2Vuc2l0aXZlLCBzbyBtaXhlZCBhbmQgdXBwZXItY2FzZSBzcGVsbGluZ3Mgc3VjaCBhcyBgUmVEYCBhbmQgYEdSRUVOYCBhcmUgYWxzbyBhY2NlcHRlZC4gR3JlZW4gaXMgdGhlIGRlZmF1bHQuIFJlcGVhdGVkIGAtQ2Agb3B0aW9ucyBhcmUgcGVybWl0dGVkLCB3aXRoIHRoZSBsYXN0IHZhbHVlIGJlY29taW5nIHRoZSBjdXJyZW50IHNlbGVjdGlvbi4gYC1yYCBtYXkgYmUgY29tYmluZWQgd2l0aCBgLUNgIGFuZCBlbmFibGVzIHZhcnlpbmcgcmFpbmJvdyBjb2xvcnMgZHVyaW5nIHJlbmRlcmluZy4KCkFuIHVucmVjb2duaXplZCBjb2xvciwgaW5jbHVkaW5nIGFuIGVtcHR5IHN0cmluZywgaXMgcmVqZWN0ZWQgaW1tZWRpYXRlbHksIGJlZm9yZSB0ZXJtaW5hbCBpbml0aWFsaXphdGlvbiBvciBsYXRlciBvcHRpb25zIGFyZSBwcm9jZXNzZWQuIFRoZSBwcm9ncmFtIGV4aXRzIHdpdGggc3RhdHVzIDAsIHdyaXRlcyBub3RoaW5nIHRvIHN0YW5kYXJkIG91dHB1dCwgYW5kIHdyaXRlcyBleGFjdGx5IHRoaXMgdHdvLWxpbmUgZGlhZ25vc3RpYyB0byBzdGFuZGFyZCBlcnJvciwgaW5jbHVkaW5nIGxlYWRpbmcgc3BhY2VzIGFuZCB0aGUgZmluYWwgbmV3bGluZToKCmBgYHRleHQKIEludmFsaWQgY29sb3Igc2VsZWN0aW9uCiBWYWxpZCBjb2xvcnMgYXJlIGdyZWVuLCByZWQsIGJsdWUsIHdoaXRlLCB5ZWxsb3csIGN5YW4sIG1hZ2VudGEgYW5kIGJsYWNrLgpgYGAKClBhcnRpYWwgbmFtZXMgYW5kIHN1YnN0cmluZ3Mgc3VjaCBhcyBgZ3JlYCBhbmQgYHJlZW5gLCBudW1lcmljIHN0cmluZ3MsIGFuZCB1bnJlbGF0ZWQgbmFtZXMgc3VjaCBhcyBgcHVycGxlYCBhcmUgaW52YWxpZC4KCiMjIFNwZWMgMTEKCmAtdSBERUxBWWAgY29uc3VtZXMgb25lIGFyZ3VtZW50IGFuZCBjb250cm9scyB1cGRhdGUgc3BlZWQuIFRoZSBkb2N1bWVudGVkIHVzZWZ1bCByYW5nZSBpcyB0aGUgaW50ZWdlciB2YWx1ZXMgMCB0aHJvdWdoIDEwLCBpbmNsdXNpdmUsIHdpdGggZGVmYXVsdCA0OyAwIGlzIGZhc3Rlc3QgYW5kIGxhcmdlciB2YWx1ZXMgc2xvdyB0aGUgdXBkYXRlLiBBbGwgZWxldmVuIGRvY3VtZW50ZWQgdmFsdWVzIG11c3QgZW50ZXIgdGhlIHJlbmRlciBsb29wLCBhbmQgYXR0YWNoZWQgc3ludGF4IHN1Y2ggYXMgYC11NGAgaXMgYWNjZXB0ZWQuIFJlcGVhdGVkIHVwZGF0ZSBvcHRpb25zIGFyZSBzYWZlIGFuZCB0aGUgbGFzdCBvbmUgY29udHJvbHMgc3Vic2VxdWVudCByZW5kZXJpbmcuCgpEZWxheSBjb252ZXJzaW9uIGZvbGxvd3MgcGVybWlzc2l2ZSBDLXN0eWxlIGRlY2ltYWwgaW50ZWdlciBwYXJzaW5nOiBhbiBpbml0aWFsIGRlY2ltYWwgcG9ydGlvbiBpcyB1c2VkLCBzbyB2YWx1ZXMgc3VjaCBhcyBgMy41YCBiZWhhdmUgbGlrZSAzIGFuZCBhIHZhbHVlIHdpdGggbm8gaW5pdGlhbCBkaWdpdHMgYmVoYXZlcyBsaWtlIDAuIERlbGF5IHBhcnNpbmcgaXRzZWxmIG11c3Qgbm90IGNyYXNoLiBJZiBhIGxhdGVyIGAtVmAgb3IgYC1oYCBpcyBlbmNvdW50ZXJlZCwgdGhhdCBpbmZvcm1hdGlvbmFsIG9wdGlvbiBzdGlsbCBwcmludHMgbm9ybWFsbHkuCgojIyBTcGVjIDEyCgpgLU0gTUVTU0FHRWAgY29uc3VtZXMgZXhhY3RseSB0aGUgbmV4dCBhcmd1bWVudCBhcyBhIG1lc3NhZ2UuIFRoZSBhcmd1bWVudCBtYXkgYmUgZW1wdHkgb3IgbWF5IGNvbnRhaW4gc3BhY2VzLCBxdW90ZXMsIHB1bmN0dWF0aW9uLCBlcXVhbHMgc2lnbnMsIHRhYnMsIG5ld2xpbmVzLCBVbmljb2RlLCBlbW9qaSwgb3IgdGhvdXNhbmRzIG9mIGNoYXJhY3RlcnMgd2l0aG91dCBjYXVzaW5nIGEgY3Jhc2guIEluIHRoZSBVSSwgYSBub25lbXB0eSBtZXNzYWdlIGlzIG92ZXJsYWlkIHZpc2libHkgYXJvdW5kIHRoZSBjZW50ZXIgb2YgdGhlIHRlcm1pbmFsOyBmb3IgZXhhbXBsZSwgYC1NICJURVNUIE1FU1NBR0UiYCBtdXN0IGxlYXZlIGBURVNUYCBvciBgTUVTU0FHRWAgdmlzaWJsZSBpbiBhIGNhcHR1cmVkIHBhbmUgd2hpbGUgcmFpbiBjb250aW51ZXMgYXJvdW5kIGl0LiBUZXh0IHdoaWNoIGRvZXMgbm90IGZpdCBtYXkgYmUgc2FmZWx5IGNsaXBwZWQgdG8gdGVybWluYWwgd2lkdGguCgpJbiBsb2NrZWQgcHJlc2VudGF0aW9uIG1vZGUsIGAtTGAgc3VwcGxpZXMgYSB2aXNpYmxlIGRlZmF1bHQgbWVzc2FnZSBjb250YWluaW5nIHdvcmRpbmcgc3VjaCBhcyBgQ29tcHV0ZXIgbG9ja2VkYDsgYSBzdXBwbGllZCBgLU1gIGlzIHRoZSBjdXN0b20gbWVzc2FnZSByYXRoZXIgdGhhbiB0aGUgZGVmYXVsdC4gTWVzc2FnZSBoYW5kbGluZyBtdXN0IHJlbWFpbiBjb21wYXRpYmxlIHdpdGggdGhlIG90aGVyIGRpc3BsYXkgYW5kIGNvbG9yIG9wdGlvbnMuCgojIyBTcGVjIDEzCgpXaXRoIGAtc2AsIHRoZSBhcHBsaWNhdGlvbiBpcyBpbiBzY3JlZW5zYXZlciBtb2RlLiBJdCBmaXJzdCBpbml0aWFsaXplcyB0aGUgYWx0ZXJuYXRlIHNjcmVlbiBhbmQgYmVnaW5zIGRyYXdpbmcgbm9ybWFsbHkuIFRoZSBmaXJzdCBrZXlib2FyZCBpbnB1dCB0aGVuIGNhdXNlcyBwcm9tcHQgdGVybWluYXRpb24gYW5kIHRlcm1pbmFsIHJlc3RvcmF0aW9uLCByZWdhcmRsZXNzIG9mIHdoaWNoIGNoYXJhY3RlciB3YXMgcHJlc3NlZDsgYm90aCBgcWAgYW5kIGFuIGFyYml0cmFyeSBwcmludGFibGUga2V5IG11c3Qgd29yay4gVGhpcyBzaHV0ZG93biBtdXN0IGJlIHF1aWNrIGVub3VnaCBmb3IgYSBjb250cm9sbGluZyB0ZXJtaW5hbCB0byBvYnNlcnZlIHRoYXQgdGhlIHByb2dyYW0gaGFzIHJldHVybmVkIHdpdGhpbiBhcHByb3hpbWF0ZWx5IDMwMCBtcy4KCiMjIFNwZWMgMTQKCkluIG9yZGluYXJ5IGludGVyYWN0aXZlIG1vZGUsIGtleWJvYXJkIGlucHV0IGlzIHBvbGxlZCB3aXRob3V0IHBhdXNpbmcgdGhlIGFuaW1hdGlvbi4gVGhlIGZvbGxvd2luZyBrZXlzIGNoYW5nZSBsaXZlIHNldHRpbmdzIGFuZCBtdXN0IG5ldmVyIHRlcm1pbmF0ZSBvciBjb3JydXB0IHRoZSByZW5kZXJlciBtZXJlbHkgYmVjYXVzZSB0aGV5IGFyZSBwcmVzc2VkOgoKLSBgYWAgdG9nZ2xlcyBhc3luY2hyb25vdXMgc2Nyb2xsaW5nOwotIGBiYCwgYEJgLCBhbmQgYG5gIHNlbGVjdCByYW5kb20tYm9sZCwgYWxsLWJvbGQsIGFuZCBuby1ib2xkIGludGVuc2l0eSBtb2RlczsKLSBkaWdpdHMgYDBgIHRocm91Z2ggYDlgIHNlbGVjdCB0aGUgY29ycmVzcG9uZGluZyB1cGRhdGUgZGVsYXk7Ci0gYCFgLCBgQGAsIGAjYCwgYCRgLCBgJWAsIGBeYCwgYW5kIGAmYCBzZWxlY3QgcmVkLCBncmVlbiwgeWVsbG93LCBibHVlLCBtYWdlbnRhLCBjeWFuLCBhbmQgd2hpdGUsIHJlc3BlY3RpdmVseTsKLSBgKWAgbGVhdmVzIHRoZSBjdXJyZW50IGJhc2UgY29sb3IgdW5jaGFuZ2VkIGFuZCB0aGUgcmVuZGVyZXIgY29udGludWVzIHJ1bm5pbmcuIEluIHBhcnRpY3VsYXIsIGl0IGRvZXMgbm90IHNlbGVjdCBBTlNJIGJsYWNrLgoKVGhlc2Uga2V5cyBtYXkgYmUgc2VudCByYXBpZGx5IG9yIGluIGFyYml0cmFyeSBzZXF1ZW5jZXMuIE90aGVyIHVuYm91bmQgaW5wdXQgaXMgaWdub3JlZCBzYWZlbHkgYW5kIHRoZSBjaGFuZ2luZyBhbmltYXRpb24gY29udGludWVzLgoKIyMgU3BlYyAxNQoKYC10IFRUWWAgY29uc3VtZXMgYSBwYXRoIG5hbWluZyB0aGUgdGVybWluYWwgZGV2aWNlL291dHB1dCB0byB1c2UuIFRoZSB2YWx1ZSBpcyBhY2NlcHRlZCBhcyBvcGFxdWUgb3B0aW9uIGRhdGEgcmF0aGVyIHRoYW4gYmVpbmcgcmVqZWN0ZWQgbWVyZWx5IGJlY2F1c2UgaXQgaXMgbm90IGl0c2VsZiBhIHRlcm1pbmFsIGRldmljZS4gUGFyc2luZyB0aGUgb3B0aW9uIGRvZXMgbm90IHRlcm1pbmF0ZSB0aGUgcHJvZ3JhbSwgYW5kIGl0IGlzIGNvbXBhdGlibGUgd2l0aCBpbmZvcm1hdGlvbmFsIGZsYWdzOyBmb3IgZXhhbXBsZSwgYC10IC9kZXYvbnVsbCAtVmAgcHJpbnRzIHZlcnNpb24gaW5mb3JtYXRpb24uIFdpdGggYSBzdXBwb3J0ZWQgYFRFUk1gLCBldmVuIGEgbm9ybWFsIGAtdCAvZGV2L251bGxgIGludm9jYXRpb24gcHJvY2VlZHMgaW50byB0aGUgcmVuZGVyIGxvb3AuIElmIHRlcm1pbmFsIGluaXRpYWxpemF0aW9uIGxhdGVyIGZhaWxzIChpbmNsdWRpbmcgaW4gYW4gaW52b2NhdGlvbiB3aGljaCBuYW1lcyBhIG5vbmV4aXN0ZW50IHBhdGgpLCBpdCBtdXN0IGZhaWwgcHJvbXB0bHkgYW5kIGdyYWNlZnVsbHkgd2l0aCBhIG5vbnplcm8gc3RhdHVzIGFuZC9vciBhbiBlcnJvciBkaWFnbm9zdGljOyBpdCBtdXN0IG5vdCBjcmFzaCBmcm9tIGludmFsaWQgbWVtb3J5IGFjY2Vzcy4KCiMjIFNwZWMgMTYKCkFsbCBpbmZvcm1hdGlvbmFsIGFuZCBhcmd1bWVudC1lcnJvciBvdXRwdXQgaXMgb3JkaW5hcnkgVVRGLTggdGV4dC4gSGVscCBhbmQgdmVyc2lvbiBvdXRwdXQgZ28gZXhjbHVzaXZlbHkgdG8gc3RhbmRhcmQgb3V0cHV0OyBpbnZhbGlkLWNvbG9yIGFuZCB0ZXJtaW5hbC1pbml0aWFsaXphdGlvbiBkaWFnbm9zdGljcyBnbyB0byBzdGFuZGFyZCBlcnJvci4gSW5mb3JtYXRpb25hbCBvdXRwdXQgbmV2ZXIgY29udGFpbnMgY3Vyc2VzIHNldHVwL2NsZWFudXAgYnl0ZXMuIFRoZSByZW5kZXIgc3RyZWFtLCBieSBjb250cmFzdCwgaW50ZW50aW9uYWxseSBjb250YWlucyB0ZXJtaW5hbCBjb250cm9sIHNlcXVlbmNlcyBhbmQgZHJhd24gY2hhcmFjdGVyIGRhdGEuCgpUaGUgZXhlY3V0YWJsZSBtdXN0IHRvbGVyYXRlIHJhcGlkIHN0YXJ0dXAgYW5kIGludGVycnVwdGlvbiwgcmVzdG9yZSB0ZXJtaW5hbCBzZXR0aW5ncyBvbiBldmVyeSBjb250cm9sbGVkIGV4aXQgcGF0aCwgYW5kIGF2b2lkIGVtaXR0aW5nIG1lc3NhZ2VzIGNvbnRhaW5pbmcgdGhlIHdvcmQgYGVycm9yYCBkdXJpbmcgc3VjY2Vzc2Z1bCBpbml0aWFsaXphdGlvbiBvZiB0aGUgc3VwcG9ydGVkIGRpc3BsYXkgbW9kZXMuCg==').decode()
df = pd.DataFrame([{ 'task': 'abishekvashok__cmatrix.5c082c6', 'spec': SPEC }])

# default model:
results = verigap.evaluate(llm=[kbench.llm], evaluation_data=df)

# multi-model side-by-side (uncomment and pick from kbench.llms):
# results = verigap.evaluate(llm=[kbench.llms['google/gemma-4-e4b'], kbench.llms['google/gemini-2.5-flash']], evaluation_data=df)

print(results.as_dataframe())